# Homework 3: The Dawn of Neural Machine Translation with Seq2Seq

## Part 1: Historical Context and Motivation

Before the rise of deep learning, machine translation (MT) was dominated by **Statistical Machine Translation (SMT)**. SMT systems were complex engineering feats, relying on statistical models to translate phrases piece-by-piece and then reassembling them using intricate rules.

In 2014, a seminal paper changed the landscape: **"Sequence to Sequence Learning with Neural Networks"** by Sutskever, Vinyals, and Le. They proposed an elegant, end-to-end neural architecture.

### The Core Idea

The core idea is remarkably simple:

1. **The Encoder**: An RNN reads the input sentence (e.g., English) one word at a time, compressing the entire meaning into a single, fixed-size vector. This is often called the **context vector** or, more poetically, a **"thought vector."**

2. **The Decoder**: Another RNN takes this "thought vector" as its starting point and generates the output sentence (e.g., French) one word at a time.

This architecture marked the beginning of **Neural Machine Translation (NMT)**. In 2016, Google Translate switched from its older SMT system to NMT. The improvement was dramatic.

> **"With this update, Google Translate is improving more in a single leap than we've seen in the last ten years combined."** – [Google Blog, 2016 ](https://blog.google/products/translate/found-translation-more-accurate-fluent-sentences-google-translate/)

The original 2014 paper used LSTMs. However, we will use **Gated Recurrent Units (GRUs)** for this assignment just to mix it up! GRUs are similar to LSTMs in that they use gates to control information flow, but their architecture is simpler (two gates vs. three, and no separate cell state). They often perform similarly to LSTMs but are slightly faster to train and easier to implement.

---

## Part 2: Key Concepts

### 2.1 Backpropagation Through Time (BPTT)

When training RNNs, we must backpropagate gradients through all time steps of the sequence. This is called **Backpropagation Through Time (BPTT)**. The gradients flow backwards through the unrolled RNN, allowing the model to learn long-term dependencies.

### 2.2 BPTT and Truncated BPTT (TBPTT)

If a sequence is very long (e.g., modeling an entire document), full BPTT consumes excessive memory because we must store the activations for every time step.

**Truncated BPTT (TBPTT)** solves this by breaking the sequence into chunks. We process a chunk, backpropagate gradients only within that chunk, and then pass the hidden state forward to the next chunk, stopping the gradient flow at the chunk boundary.

In this assignment, our sentences are short, so we will use standard BPTT.

### 2.3 The "Reversal Trick"

The 2014 paper discovered a surprising trick that significantly boosted performance: **Reverse the source sentence.**

- **Original**: [I, love, AI] → [J'aime, l'IA]
- **Reversed**: [AI, love, I] → [J'aime, l'IA]

By doing this, the first words of the output (J'aime) are very close to the corresponding words in the reversed input (I). This creates short-term dependencies, making it much easier for the optimizer to "establish communication" between the input and the output early in training.

### 2.4 Teacher Forcing

When training the decoder, if we use the model's prediction as the input for the subsequent step, an early mistake can cascade, making training unstable.

**Teacher Forcing** is a strategy where we sometimes use the actual ground truth token from the training data as the input for the next step, rather than the model's own prediction.

---

## Part 3: Setup and Data Preprocessing

We will use a dataset of English-French sentence pairs.

### 3.0 Download the Data

See `runner.py`.

### 3.1 Imports and Utilities (Provided)

See `runner.py`.

### 3.2 Vocabulary and Data Loading (Provided)

We provide the utilities to load, normalize, and filter the data. We limit the dataset size and sentence length for faster training.

See `runner.py`.

### 3.3 Dataset and DataLoader (Provided)

We implement the PyTorch Dataset. This is where we apply the **Input Reversal Trick**.

We also implement a `collate_fn`. This function handles padding sequences in a batch to the same length. Crucially, it also returns the original lengths of the sequences, which we need for **Packing**.

See `runner.py`.

---

## Part 4: The Seq2Seq Architecture (Implementation Tasks)

Now we implement the core components.

### Task 1: The Encoder (20 Points)

The Encoder processes the input sequence and compresses it into the context vector.

**Important: Packing Padded Sequences.** When training RNNs on batches, we must use `pack_padded_sequence`. This tells the GRU/LSTM to ignore PAD tokens. If we don't pack, the RNN processes the padding, which wastes computation and can negatively affect the final hidden state (the context vector).

**Instructions:**

1. Initialize the `nn.Embedding` and `nn.GRU` layers. Use `batch_first=True`.
2. In the forward pass, embed the input.
3. Pack the embedded sequence using `pack_padded_sequence`.
4. Pass the packed sequence through the GRU.
5. Return the final hidden state.

See `EncoderGRU.py`.

### Task 2: The Decoder (20 Points)

The Decoder takes the context vector as its initial hidden state and generates the output sequence one token at a time.

**Instructions:**

1. Initialize the Embedding, GRU, and output Linear (`fc_out`) layers.
2. The forward pass accepts one token (`input`) and the previous hidden state.
3. Embed the input token (remembering to add a sequence dimension).
4. Pass the embedding and hidden state to the GRU.
5. Pass the GRU output through the linear layer to get the prediction logits.
6. Return the prediction and the new hidden state.

See `DecoderGRU.py`.

### Task 3: The Seq2Seq Wrapper (30 Points)

This class combines the Encoder and Decoder and manages the overall process, including the decoding loop and Teacher Forcing.

**Instructions:**

1. Run the encoder on the source sequence and lengths to get the context vector (hidden).
2. Initialize the decoder input with the `<SOS>` token.
3. Iterate over the length of the target sequence:
    - Run the decoder one step.
    - Store the output.
    - Decide whether to use teacher forcing or the model's own prediction as the next input.

See `Seq2Seq.py`.

---

## Part 5: Training the Model

### 5.1 Initialization (Provided)

We initialize the model with sensible hyperparameters. We use a relatively small model (2 layers, 512 hidden units) which provides a good balance of capacity and training speed for this dataset.

See `runner.py`.

### Task 4: The Training and Evaluation Loops (20 Points)

Implement the training and evaluation functions.

**Instructions:**

1. In `train`, implement the forward pass, loss calculation, backpropagation (BPTT), gradient clipping, and optimizer step.
2. In `evaluate`, implement the forward pass (with `teacher_forcing_ratio=0`).
3. **Crucial:** Reshape the output and tgt tensors correctly for the loss function. CrossEntropyLoss expects predictions of shape `(N, C)` and targets of shape `(N)`, where N is the total number of tokens.

See `train.py`.

See `evaluate.py`.

### 5.3 Running the Training (Provided)

See `runner.py`.

**Important Note on Training Duration:**

Vanilla Seq2Seq models typically require 30-50+ epochs to produce reasonable translations. If you train for only 15 epochs, you will likely see:
- Decreasing loss (showing the model is learning)
- But poor actual translations (the model hasn't converged yet)

This is normal! The model needs more time to learn the complex mapping between languages.

---

## Part 6: Inference and Analysis (10 Points)

### 6.1 Inference (Greedy Decoding)

During inference, we don't have the target sentence, so teacher forcing is impossible. We use the model's own predictions at each step. The simplest method is **Greedy Decoding**: always choose the word with the highest probability.

See `runner.py`.

### 6.2 Understanding Your Results

**Expected Performance:**

After training, you may notice that your translations are not perfect - and that's completely normal! Here's what you should expect:

**What Good Results Look Like:**
- Training loss decreasing from ~5.0 to ~1.0-1.5
- Validation loss around 2.5-3.5
- Some simple phrases translating correctly (e.g., "how are you" → "comment vas tu")
- Shorter sentences working better than longer ones

**Why Translations May Be Poor:**

1. **The Information Bottleneck**: This is the fundamental limitation we've been discussing. The entire English sentence must be compressed into a single fixed-size vector (512 numbers). For complex sentences, critical information gets lost.

2. **Insufficient Training**: 30 epochs on this small dataset is barely enough. Production NMT systems train for much longer on millions of examples.

3. **Overfitting**: If your validation loss is significantly higher than training loss (e.g., 2.8 vs 1.3), the model is memorizing training patterns rather than learning to translate.

4. **Common Phrase Bias**: The model often outputs frequent French phrases (like "je suis...") regardless of the actual input, because these patterns were common in training data.

5. **Greedy Decoding**: We always pick the highest probability word. Beam search (which considers multiple possibilities) would improve results.

**What Your Model Is Actually Learning:**

Look at a translation like:
```
"i am cold" → "je suis serieux"
```

The model correctly learned:
- "i am" → "je suis" ✓
- But outputs a common word "serieux" instead of "froid"

This shows the model IS learning French grammar and common patterns, just not the specific vocabulary mapping yet.

**This Is Why Attention Was Invented!**

The poor performance of vanilla Seq2Seq on longer sentences directly motivated the invention of attention mechanisms (covered in the next module). Attention allows the decoder to "look back" at different parts of the input instead of relying on a single compressed vector.

---

### 6.3 Bonus: Diagnostic Function (Optional)

To better understand what your model has learned, implement this diagnostic function that checks if the model can at least memorize some training examples:

See `runner.py`.

If the model can't even memorize training examples with >50% word overlap, it needs more training epochs or there may be a bug.

---

### 6.4 Conceptual Questions

Answer the following questions in the following text cells:

1. **The Information Bottleneck**: The core limitation of this architecture is that the encoder must compress the entire input sentence into a single fixed-size context vector (hidden). Why is this a significant problem when translating very long or complex sentences?

**Why compression is a significant problem**: The encoding is a fixed state while the information is being streamed in, so the model must learn to compress all the information into that fixed-size vector without knowing how long the input sequence will be, which promotes incredbly lossy and parsimonious representations of the inputs to be stored especially for variable length sequences where the optimal compression signficantly differs.

2. **Input Reversal**: Explain again, in your own words, why reversing the input (the "Reversal Trick") helped the model learn more effectively. Relate your answer to the concept of gradient flow in BPTT.

**Why the Reversal Trick helps the model learn**: Due to the nature of BPTT, the first token of the output will be made right after reading the last token of the input, by reversing we move the input of the first token closer to the output of its corresponding token output token. While the temporal distance between the middle tokens remains relatively constant and the last tokens much farther, the autoregressive nature of the decoder and the nature of NLP tasks means that the first few tokens are often more important for establishing the overall structure of the sentence, even if the network retains less information about the later tokens. This loss of information is due to BPTT, the gradients of later tokens in the sequence need to flow through more timesteps which even when stable can lead to more lossy representations of the input information.

3. **TBPTT Application**: While we used standard BPTT here, describe a different NLP task where Truncated BPTT (TBPTT) would be essential, and explain why standard BPTT would be unsuitable in that scenario.

**Where TBPTT is essential**: TBPTT is essential when sequence lengths are very long as it is difficult to keep the gradients stable throughout the entire sequence.

**Why BPTT would be unsuitable in that scenario**: BPTT already experience catastrophic forgetting due the lossy representations of the input information which need to be kept stable throughout the entire sequence. On very long sequences the gradients will almost always become unstable, truncating the sequence, while not allowing the model to consider the entire sequence at every token, makes training possible.

4. **Packing**: Why is it important to use `pack_padded_sequence` in the encoder when dealing with batched inputs? What might happen if we didn't use it?

**Why using `pack_padded_sequence` is important**: Padding the sequences allows us to process batches of variable length sequences, however the model must not process the padded tokens as they do not contain any information and can negatively affect the final hidden state (the context vector) if processed by the GRU. Packing the sequence allows the GRU to ignore the padded tokens and only process the actual input tokens, which leads to a more accurate hidden state.

**What might happen if `pack_padded_sequence` was not used**: The model gradients during BPTT may become unstable due to the noise introduced by the padded tokens leading to catastrophic forgetting in the forward pass.

For those that are interested to improve the performance, try to add:[optional]
- Beam Search for better decoding (instead of greedy)
- Better evaluation metrics (BLEU score)